# ETL_02 — Poblar `dim_variables_etf` (v2)
* **Proyecto:** INGEI AFOLU – DNCC / MADES – Paraguay  
* **Propósito:** Parsear el `metadata.json.lzma` del etf-cli y cargar los UIDs  
de variables AFOLU (Agriculture + LULUCF) en `inventario.dim_variables_etf`.  
Esta tabla es el **puente central** del pipeline: código IPCC + medida + gas → `variable_uid`.

* **Input:**  `etf-cli/src/unfccc/etf/assets/metadata.json.lzma`  
* **Output:** `inventario.dim_variables_etf`
* **Desarrollo:** Claudia Palacios | cpalacios01@gmail.com | +595994648273

---


## 0. Configuración

In [ ]:
from pathlib import Path

# ── Rutas ───────────────────────────────────────────────────
BASE_DIR      = Path('.')                              # Dev_Cp/
METADATA_PATH = BASE_DIR / 'etf-cli' / 'src' / 'unfccc' / 'etf' / 'assets' / 'metadata.json.lzma'
OUT_DIR       = BASE_DIR / 'out'
OUT_DIR.mkdir(exist_ok=True)

# ── Conexión ────────────────────────────────────────────────
DB_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'dbname':   'ingei_afolu',
    'user':     'postgres',
    'password': 'password'
}
DB_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
)

# ── Parámetros ──────────────────────────────────────────────
METADATA_VER   = '1.30.4'    # versión del metadata ETF
BATCH_SIZE     = 1000        # filas por batch de inserción
SECTORES_AFOLU = {
    '43bc1534-201c-416b-a348-e5866d69dddb': 'agriculture',
    'db7b9be0-76bc-497e-a4ee-9334ec2429d2': 'lulucf'
}

print(f'Metadata : {METADATA_PATH}')
print(f'Existe   : {METADATA_PATH.exists()}')
print(f'Output   : {OUT_DIR.resolve()}')

## 1. Imports

In [ ]:
import lzma
import json
import re
import time
import uuid as uuid_lib
import pandas as pd
from sqlalchemy import create_engine, text, String

engine = create_engine(DB_URL)
print('✅ Imports OK — engine creado')

## 2. Cargar metadata ETF

In [ ]:
t0 = time.time()
print('Cargando metadata.json.lzma...')

with lzma.open(METADATA_PATH, 'rb') as f:
    meta = json.load(f)

root      = meta['Metadata'][0]
variables = root['variable']
ms = round((time.time() - t0) * 1000)

print(f'✅ Metadata cargada en {ms} ms')
print(f'   Versión      : {root["version"]["name"]}')
print(f'   Publicada    : {root["version"]["publication_date"]}')
print(f'   Total vars   : {len(variables):,}')

## 3. Filtrar variables AFOLU

In [ ]:
vars_afolu = [
    v for v in variables
    if v.get('role_node_uid') in SECTORES_AFOLU
]

agri   = sum(1 for v in vars_afolu if v.get('role_node_uid') == '43bc1534-201c-416b-a348-e5866d69dddb')
lulucf = sum(1 for v in vars_afolu if v.get('role_node_uid') == 'db7b9be0-76bc-497e-a4ee-9334ec2429d2')

print(f'Variables AFOLU total : {len(vars_afolu):,}')
print(f'  Agriculture (3.x)   : {agri:,}')
print(f'  LULUCF     (4.x)    : {lulucf:,}')

## 4. Parsear nombres de variables

Cada nombre tiene el formato `[categoría][subcategoría][medida][gas][parámetro][unidad]`.  
Extraemos cada componente y normalizamos el gas al código canónico.


In [ ]:
# ── Patrones ────────────────────────────────────────────────
BRACKET_PAT = re.compile(r'\[([^\]]+)\]')
CODE_PAT    = re.compile(
    r'^(\d+(?:\([A-Z]+\))?\.?[A-Z]?\d*(?:\.\d+)*(?:\.[a-z](?:\.[a-z])?(?:\.[a-z])?)?)'
)

# ── Normalización de gases ──────────────────────────────────
GAS_NORM = {
    'CO\u2082': 'CO2',
    'CH\u2084': 'CH4',
    'N\u2082O': 'N2O',
    'Aggregate GHGs': 'GHG_AGG',
    'no gas': None,
    'HFCs': 'HFC',
    'PFCs': 'PFC',
    'SF\u2086': 'SF6',
    'NF\u2083': 'NF3',
}
GAS_SET = set(GAS_NORM.keys())

# ── Palabras clave de medida ────────────────────────────────
MEASURE_KEYWORDS = {
    'Emissions', 'Activity Data', 'Implied emission factor',
    'Emission factor information', 'Carbon stock change',
    'Indirect emissions', 'Annual change in stock', 'Area',
    'Population', 'Total Emissions', 'Source emissions',
    'Amount applied', 'Gains', 'Losses', 'Production',
    'Harvested area', 'Total area', 'Area burned',
    'Biomass burned', 'Nitrogen input', 'Total N excreted',
    'Net CO2 emissions/ removals from HWP in use',
    'Annual domestic harvest', 'Annual exports of wood, and paper products + wood fuel, pulp, recovered paper, roundwood/chips',
    'Annual imports of wood, and paper products + wood fuel, pulp, recovered paper, roundwood/chips',
}

def parse_variable(v: dict, sector: str) -> dict:
    """Parsea un objeto variable del metadata ETF y retorna un dict para la BD.

    v2: nunca retorna None en columnas NOT NULL (variable_uid, nombre_completo, medida, sector).
    Casos límite cubiertos:
      - Nombre con 1 solo bracket (ej. nodos de cálculo intermedio) → medida = ese bracket.
      - Nombre sin brackets en absoluto → medida = 'Sin clasificar'.
      - Nombre ausente/vacío → nombre_completo con placeholder trazable por uid.
    """
    name    = v.get('name') or ''
    parts   = BRACKET_PAT.findall(name)
    cat_raw = parts[0] if parts else ''
    mc      = CODE_PAT.match(cat_raw)

    gas_raw = next((p for p in parts if p in GAS_SET), None)

    medida_fallback = False
    encontrada = next((p for p in parts if p in MEASURE_KEYWORDS), None)
    if encontrada is not None:
        medida = encontrada
    elif len(parts) > 2:
        medida = parts[2]
    elif len(parts) >= 1:
        # nodos de cálculo intermedio u otros formatos cortos: "[Intermediate calc]", etc.
        medida = cat_raw
        medida_fallback = True
    else:
        # caso límite: nombre sin brackets en absoluto
        medida = 'Sin clasificar'
        medida_fallback = True

    unidad    = parts[-1] if parts else None
    parametro = parts[-2] if len(parts) > 1 else None

    return {
        'variable_uid'     : v.get('uid'),
        'codigo_ipcc'      : mc.group(1) if mc else None,
        'nombre_completo'  : (name[:500] if name else f"[sin nombre] uid={v.get('uid')}"),
        'medida'           : medida,
        'gas_codigo'       : GAS_NORM.get(gas_raw) if gas_raw else None,
        'parametro'        : parametro,
        'unidad'           : unidad,
        'sector'           : sector,
        'es_calculada'     : v.get('is_calculated', False),
        'es_editable'      : v.get('is_editable', True),
        'nke_permitido'    : v.get('nke_allowed', True),
        'node_uid'         : v.get('node_uid'),
        'metadata_ver'     : METADATA_VER,
        '_medida_fallback' : medida_fallback,   # bandera QA — se descarta antes de insertar en BD
    }

# ── Parsear todos ───────────────────────────────────────────
t0 = time.time()
records = [
    parse_variable(v, SECTORES_AFOLU[v['role_node_uid']])
    for v in vars_afolu
]
ms = round((time.time() - t0) * 1000)

df = pd.DataFrame(records)
n_fallback = int(df['_medida_fallback'].sum())

print(f'✅ Parseo completado en {ms} ms')
print(f'   Filas generadas       : {len(df):,}')
print(f'   Con código IPCC       : {df["codigo_ipcc"].notna().sum():,}')
print(f'   Sin código IPCC       : {df["codigo_ipcc"].isna().sum():,}  (subcategorías/desagregaciones)')
print(f'   Editables             : {df["es_editable"].sum():,}')
print(f'   Calculadas            : {df["es_calculada"].sum():,}')
print(f'   Con medida por fallback: {n_fallback:,}  (nodos de cálculo intermedio / nombre corto)')

## 5. Validación de integridad

In [ ]:
NOT_NULL_COLS = ['variable_uid', 'nombre_completo', 'medida', 'sector']

def es_uuid_valido(x):
    if x is None:
        return False
    try:
        uuid_lib.UUID(str(x))
        return True
    except (ValueError, TypeError, AttributeError):
        return False

problemas = []

for col in NOT_NULL_COLS:
    n_nulos = df[col].isna().sum()
    if n_nulos > 0:
        problemas.append(f'{n_nulos} filas con {col} nulo')

mask_uid_invalido = ~df['variable_uid'].apply(es_uuid_valido)
n_uid_invalido = int(mask_uid_invalido.sum())
if n_uid_invalido > 0:
    problemas.append(f'{n_uid_invalido} filas con variable_uid con formato UUID inválido')

mask_node_invalido = df['node_uid'].notna() & ~df['node_uid'].apply(es_uuid_valido)
n_node_invalido = int(mask_node_invalido.sum())
if n_node_invalido > 0:
    problemas.append(f'{n_node_invalido} filas con node_uid con formato UUID inválido (se normalizarán a NULL en la sección 9)')

n_duplicados = int(df['variable_uid'].duplicated().sum())
if n_duplicados > 0:
    problemas.append(f'{n_duplicados} variable_uid duplicados (violaría UNIQUE)')

print('── Validación de integridad ──')
if problemas:
    for p in problemas:
        print(f'  ⚠️  {p}')
else:
    print('  ✅ Sin problemas detectados')

# ── Cuarentena: filas que NO se pueden insertar de forma segura ──
mask_cuarentena = (
    df['variable_uid'].isna() |
    mask_uid_invalido |
    df['variable_uid'].duplicated(keep='first')
)
df_cuarentena = df[mask_cuarentena].copy()
df = df[~mask_cuarentena].copy()

print()
print(f'  Filas válidas para insertar : {len(df):,}')
print(f'  Filas en cuarentena         : {len(df_cuarentena):,}')

if len(df_cuarentena) > 0:
    cuarentena_path = OUT_DIR / 'dim_variables_etf_cuarentena.csv'
    df_cuarentena.to_csv(cuarentena_path, index=False, encoding='utf-8-sig')
    print(f'  📄 Detalle guardado en: {cuarentena_path}')

## 6. Explorar el DataFrame parseado

In [ ]:
print('Distribución por sector:')
print(df['sector'].value_counts().to_string())
print()
print('Distribución por gas:')
print(df['gas_codigo'].value_counts(dropna=False).to_string())
print()
print('Distribución por medida (top 10):')
print(df['medida'].value_counts().head(10).to_string())

In [ ]:
# Muestra de filas con código IPCC — las más importantes para el pipeline
df[df['codigo_ipcc'].notna()][[
    'codigo_ipcc','sector','medida','gas_codigo','unidad','es_editable','variable_uid'
]].head(15)

## 7. Exportar CSV (respaldo en `out/`)

In [ ]:
csv_path = OUT_DIR / 'dim_variables_etf.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'✅ CSV guardado: {csv_path}')
print(f'   Tamaño: {csv_path.stat().st_size / 1024:.0f} KB')
print(f'   (incluye columna _medida_fallback para trazabilidad QA; se descarta antes del insert)')

## 8. Verificar estado actual de la tabla en BD

In [ ]:
with engine.connect() as con:
    count_actual = con.execute(
        text('SELECT COUNT(*) FROM inventario.dim_variables_etf')
    ).scalar()

print(f'Filas actuales en dim_variables_etf : {count_actual:,}')

if count_actual > 0:
    print()
    print('⚠️  La tabla ya tiene datos.')
    print('   Si querés recargar, ejecutá la celda de limpieza (sección 8b) antes de continuar.')
else:
    print('✅ Tabla vacía — lista para la carga inicial.')

In [ ]:
# ── SECCIÓN 8b: LIMPIEZA (solo si necesitás recargar) ──────
# Descomentar y ejecutar SOLO si querés borrar y reinsertar todo

with engine.begin() as con:
     con.execute(text('TRUNCATE TABLE inventario.dim_variables_etf RESTART IDENTITY CASCADE'))
print('⚠️  Tabla truncada. Proceda con la carga.')

## 9. Limpiar y normalizar UUIDs

In [ ]:
def limpiar_uuid(x):
    """Devuelve un string UUID normalizado, o None si está vacío / es inválido."""
    if x is None or (isinstance(x, str) and x.strip() == ''):
        return None
    try:
        return str(uuid_lib.UUID(str(x)))
    except (ValueError, TypeError, AttributeError):
        return None

antes_node_nulos = int(df['node_uid'].isna().sum())

df['variable_uid'] = df['variable_uid'].apply(limpiar_uuid)
df['node_uid']      = df['node_uid'].apply(limpiar_uuid)

despues_node_nulos = int(df['node_uid'].isna().sum())
node_normalizados_a_null = despues_node_nulos - antes_node_nulos

print(f'variable_uid nulos : {df["variable_uid"].isna().sum()}  (debe ser 0)')
print(f'node_uid nulos      : {despues_node_nulos:,}')
if node_normalizados_a_null > 0:
    print(f'  ↳ {node_normalizados_a_null} node_uid con formato inválido normalizados a NULL')
print('✅ Columnas UUID limpias y normalizadas — listo para insertar')

assert df['variable_uid'].isna().sum() == 0, 'variable_uid no puede tener nulos — revisar cuarentena (sección 5)'

## 10. Insertar en `inventario.dim_variables_etf`

Carga en batches de 1.000 filas. **Nota:** si un batch falla como bloque, se reintenta
fila por fila para aislar el problema — así una única fila inesperada no vuelve a tirar abajo
las otras ~999 filas sanas del batch.

In [ ]:
import math

DB_COLS = [
    'variable_uid','codigo_ipcc','nombre_completo','medida','gas_codigo',
    'parametro','unidad','sector','es_calculada','es_editable',
    'nke_permitido','node_uid','metadata_ver'
]
df_insert = df[DB_COLS].copy()   # descarta columnas de QA (_medida_fallback) que no existen en la tabla

total   = len(df_insert)
batches = math.ceil(total / BATCH_SIZE)
t0      = time.time()
filas_con_error = []
filas_ok        = 0
batches_con_retry = 0

print(f'Insertando {total:,} filas en {batches} batches de {BATCH_SIZE}...')
print()

for i in range(batches):
    chunk = df_insert.iloc[i * BATCH_SIZE : (i + 1) * BATCH_SIZE].copy()
    try:
        chunk.to_sql(
            name       = 'dim_variables_etf',
            schema     = 'inventario',
            con        = engine,
            if_exists  = 'append',
            index      = False,
            method     = 'multi',
            dtype      = {
                'variable_uid': String,   # SQLAlchemy lo trata como texto; Postgres lo castea a UUID
                'node_uid'    : String,
            }
        )
        filas_ok += len(chunk)
    except Exception as e:
        batches_con_retry += 1
        print(f'  ⚠️  Batch {i} falló como bloque ({str(e).splitlines()[0][:120]})')
        print(f'      Reintentando fila por fila...')
        for idx, row in chunk.iterrows():
            fila = pd.DataFrame([row])
            try:
                fila.to_sql(
                    name='dim_variables_etf', schema='inventario', con=engine,
                    if_exists='append', index=False,
                    dtype={'variable_uid': String, 'node_uid': String}
                )
                filas_ok += 1
            except Exception as e2:
                filas_con_error.append({
                    'variable_uid'   : row.get('variable_uid'),
                    'nombre_completo': row.get('nombre_completo'),
                    'error'          : str(e2).splitlines()[0][:200]
                })

    if (i + 1) % 5 == 0 or (i + 1) == batches:
        pct     = round((i + 1) / batches * 100)
        elapsed = round(time.time() - t0, 1)
        print(f'  Batch {i+1:>3}/{batches} — {pct:>3}% — {elapsed}s')

elapsed_total = round(time.time() - t0, 1)
print()
print(f'✅ Filas insertadas  : {filas_ok:,} / {total:,}')
if batches_con_retry:
    print(f'   Batches con retry : {batches_con_retry}')
if filas_con_error:
    print(f'❌ Filas con error   : {len(filas_con_error):,}')
    errores_path = OUT_DIR / 'dim_variables_etf_errores_insercion.csv'
    pd.DataFrame(filas_con_error).to_csv(errores_path, index=False, encoding='utf-8-sig')
    print(f'   Detalle guardado en: {errores_path}')
else:
    print(f'   Sin errores de inserción.')
print(f'   Tiempo total: {elapsed_total}s')

## 11. Verificación post-carga

In [ ]:
SQL_VERIFY = """
SELECT
    sector,
    COUNT(*)                                        AS total_vars,
    COUNT(*) FILTER (WHERE codigo_ipcc IS NOT NULL) AS con_codigo_ipcc,
    COUNT(*) FILTER (WHERE es_editable = TRUE)      AS editables,
    COUNT(*) FILTER (WHERE es_calculada = TRUE)     AS calculadas,
    COUNT(DISTINCT gas_codigo)                      AS gases_distintos
FROM inventario.dim_variables_etf
GROUP BY sector
ORDER BY sector
"""
df_verify = pd.read_sql(SQL_VERIFY, engine)
print('Resumen por sector:')
print(df_verify.to_string(index=False))
print()

total_bd = df_verify['total_vars'].sum()
if total_bd == len(df_insert):
    print(f'✅ Conteo coincide: {total_bd:,} filas en BD = {len(df_insert):,} filas válidas parseadas')
else:
    print(f'⚠️  Discrepancia: BD={total_bd:,} vs válidas parseadas={len(df_insert):,} (revisar CSV de errores/cuarentena si corresponde)')

In [ ]:
# Verificar que los UIDs clave del 1BTR están presentes
# (tomamos algunos UIDs reales de los JSON de Energy/IPPU/Waste como referencia cruzada)
# y verificamos UIDs de Agricultura conocidos del metadata

SQL_SAMPLE = """
SELECT codigo_ipcc, sector, medida, gas_codigo, unidad, es_editable
FROM inventario.dim_variables_etf
WHERE codigo_ipcc IN ('3.A.1.a','3.A.1.b','3.B.1','3.C.7','3.D.1.a',
                      '4.A.1','4.A.2','4.B','4.C','4.D','4.E','4.F')
  AND medida = 'Emissions'
  AND gas_codigo IS NOT NULL
ORDER BY codigo_ipcc, gas_codigo
LIMIT 25
"""
df_sample = pd.read_sql(SQL_SAMPLE, engine)
print(f'Variables de emisión para categorías AFOLU clave ({len(df_sample)} resultados):')
df_sample

## 12. Registrar en auditoría

In [ ]:
import json as json_lib

detalle = {
    'metadata_ver'            : METADATA_VER,
    'total_insertadas'        : int(total_bd),
    'agriculture'             : int(df_verify[df_verify['sector']=='agriculture']['total_vars'].sum()),
    'lulucf'                  : int(df_verify[df_verify['sector']=='lulucf']['total_vars'].sum()),
    'con_codigo_ipcc'         : int(df_insert['codigo_ipcc'].notna().sum()),
    'filas_en_cuarentena'     : int(len(df_cuarentena)),
    'filas_con_medida_fallback': int(n_fallback),
    'node_uid_normalizados_a_null': int(node_normalizados_a_null),
    'batches_con_retry'       : int(batches_con_retry),
    'filas_con_error_insercion': int(len(filas_con_error)),
    'tiempo_seg'              : elapsed_total,
}

with engine.begin() as con:
    con.execute(text("""
        INSERT INTO auditoria.log_pipeline
            (etapa, resultado, mensaje, detalle)
        VALUES
            ('ETL_02_dim_variables_etf', :resultado, :mensaje, CAST(:detalle AS jsonb))
    """), {
        'resultado': 'ok' if not filas_con_error else 'warning',
        'mensaje'  : f'dim_variables_etf poblada: {total_bd:,} variables AFOLU cargadas',
        'detalle'  : json_lib.dumps(detalle),
    })

print('✅ Registro de auditoría guardado')
print(f'   {json_lib.dumps(detalle, indent=3)}')

## 13. Resumen final

In [ ]:
print('=' * 58)
print('  RESUMEN ETL_02 (v2) — dim_variables_etf')
print('=' * 58)
print(f'  Metadata ETF versión   : {METADATA_VER}')
print(f'  Variables cargadas     : {total_bd:,}')
print(f'    Agriculture (3.x)    : {detalle["agriculture"]:,}')
print(f'    LULUCF      (4.x)    : {detalle["lulucf"]:,}')
print(f'  Con código IPCC        : {detalle["con_codigo_ipcc"]:,}')
print(f'  En cuarentena          : {detalle["filas_en_cuarentena"]:,}')
print(f'  Medida por fallback    : {detalle["filas_con_medida_fallback"]:,}')
print(f'  Errores de inserción   : {detalle["filas_con_error_insercion"]:,}')
print(f'  Tiempo total           : {elapsed_total}s')
print(f'  CSV respaldo           : out/dim_variables_etf.csv')
print()
print('  Siguiente paso → ETL_03_cargar_json_sectores.ipynb')
print('  Carga Energy/IPPU/Waste 1990-2021 en fact_valores_etf')
print('=' * 58)

engine.dispose()
print('\n  Conexión cerrada.')